<!--
Copyright (c) 2026 OceanBase.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 12 · 让另一个项目用上团队经验

订单项目验证了一条金额处理经验，另一个项目也需要它。我们先观察隔离，再比较两种明确的共享方式：引用项目的上下文，以及把一个准确版本发布过去。

适合已完成 03、06 的读者；无需模型。完成后，你能解释组织层级、上下文引用和版本快照各自解决什么问题。本篇使用 Experience；当前发布接口不支持把 Memory 按同样方式发布。

路线：建立三个项目 → 验证隔离 → 引用 → 发布 → 修改原件 → 比较两个接收方。

In [ ]:
import sys
from pathlib import Path

from _tutorial import Tutorial, show, table

from powercontext.http import CreateScopeRequest

if not Path("_tutorial.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "examples" / "jupyter"))
if previous_lab := globals().get("lab"):
    await previous_lab.close()
lab = await Tutorial.start("12", features=())
client = lab.client
assert client is not None
scope = await client.create_scope(
    CreateScopeRequest(
        title="订单 CSV 导入器 · 12", summary="本次教学实验的独立材料", idempotency_key=f"{lab.run_id}:main"
    )
)
scope_id = scope.scope_id

## 先给经验找到归属

我们创建两个接收项目。一个持续引用订单项目的上下文，另一个只接受一次明确发布。先不建立引用关系，看看父子项目是否会自动共享。

In [ ]:
from powercontext.http import ArtifactReference, CreateArtifactRequest, PrepareContextRequest

content = {
    "situation": "amount 转换会截断三位小数",
    "action": "先检查精度，再转分",
    "outcome": "1.999 被拒绝",
    "lesson": "amount: 转分之前必须检查最多两位小数。",
}
created = await client.create_artifact(
    scope_id, CreateArtifactRequest.model_validate({"family": "experience", "content": content})
)
original = ArtifactReference(family="experience", artifact_id=created.artifact_id, revision=created.revision)
reader = await client.create_scope(
    CreateScopeRequest(
        title="引用团队经验的项目",
        summary="随源项目更新",
        parent_scope_id=scope_id,
        idempotency_key=f"{lab.run_id}:reader",
    )
)
recipient = await client.create_scope(
    CreateScopeRequest(
        title="接收确定版本的项目", summary="保留本次认可的快照", idempotency_key=f"{lab.run_id}:recipient"
    )
)
empty = await client.prepare_context(PrepareContextRequest(scope_id=reader.scope_id, query="amount"))
assert empty.status == "empty"
print("父子关系已存在，但子项目尚未召回父项目经验。")

## 显式引用：让准备上下文时纳入另一个项目

Scope 更新需要当前版本，避免覆盖别人刚改过的项目配置。引用关系由调用方明确设置；它不会创建一份新的 Experience。

In [ ]:
from powercontext.http import UpdateScopeRequest

reader = await client.update_scope(
    reader.scope_id,
    UpdateScopeRequest(
        expected_version=reader.version,
        title=reader.title,
        summary=reader.summary,
        parent_scope_id=reader.parent_scope_id,
        context_references=[scope_id],
    ),
)
referenced = await client.prepare_context(
    PrepareContextRequest(scope_id=reader.scope_id, query="amount", max_bytes=6000)
)
assert referenced.content and content["lesson"] in referenced.content
print(referenced.content)

## 发布：接收方获得一个可独立读取的准确版本

这里的发布目标是另一个 Scope。我们带上源项目、制品 ID 和 Revision，重复提交同一幂等请求，应得到同一份发布结果。

In [ ]:
from powercontext.http import ArtifactAddress, PublishArtifactRequest

request = PublishArtifactRequest(
    source=ArtifactAddress(scope_id=scope_id, artifact=original),
    target_scope_id=recipient.scope_id,
    idempotency_key=f"{lab.run_id}:publish",
)
publication = await client.publish_artifact(request)
assert await client.publish_artifact(request) == publication
copy = await client.get_artifact_revision(
    recipient.scope_id, "experience", publication.target.artifact.artifact_id, publication.target.artifact.revision
)
assert copy.content == content
show(publication)

## 原项目继续改进，两种共享会怎样变化？

通过 HTTP 获取真实 ETag，再补充有限值校验。我们没有改动接收方：引用方应读到源项目的新内容，发布方仍保留原快照。

In [ ]:
import httpx

new_content = {**content, "lesson": "amount: 转分前检查最多两位小数，并拒绝 NaN 等非有限值。"}
path = f"/v1/scopes/{scope_id}/artifacts/experience/{original.artifact_id}"
async with httpx.AsyncClient(base_url=lab.base_url) as http:
    current = await http.get(path)
    current.raise_for_status()
    changed = await http.put(path, headers={"If-Match": current.headers["etag"]}, json={"content": new_content})
    changed.raise_for_status()
following = await client.prepare_context(
    PrepareContextRequest(scope_id=reader.scope_id, query="amount", max_bytes=6000)
)
frozen = await client.prepare_context(
    PrepareContextRequest(scope_id=recipient.scope_id, query="amount", max_bytes=6000)
)
assert following.content and new_content["lesson"] in following.content
assert frozen.content and content["lesson"] in frozen.content and new_content["lesson"] not in frozen.content
table([{"接收方式": "显式上下文引用", "包含新校验": True}, {"接收方式": "准确版本发布", "包含新校验": False}])

## 练习与验收

把 `reader.context_references` 清空后再次准备上下文，预期恢复为空。发布到 recipient 的副本应仍可读取。成功依据是两边实际内容不同，而不是只看到发布请求返回成功。

接下来阅读 [13_team_access_control.ipynb](13_team_access_control.ipynb)。

最后关闭服务。实验文件保留在本次 `.powercontext/` 目录，便于复查。

In [ ]:
await lab.close()
print("本篇 Server 已关闭。")